In [2]:
pip install tensorflow

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.5/351.2 MB 5.6 MB/s eta 0:01:03
   ---------------------------------------- 2.4/351.2 MB 8.2 MB/s eta 0:00:43
   ---------------------------------------- 3.9/351.2 MB 8.2 MB/s eta 0:00:43
    --------------------------------------- 5.8/351.2 MB 8.2 MB/s eta 0:00:43
    --------------------------------------- 7.6/351.2 MB 8.2 MB/s eta 0:00:42
   - -------------------------------------- 9.2/351.2 MB 8.2 MB/s eta 0:00:42
   - -------------------------------------- 11.0/351.2 MB 8.2 MB/s eta 0:00:42
   - -------------------------------------- 12.6/351.2 MB 8.2 MB/s eta 0:00:42
   - -------------------------------------- 14.4/351.2 MB 8.2 MB/s eta 0:00:41
   - -------------------------------------- 16.3/351.2 MB 8.2 MB/s eta 0:00:41
   -- ------------------------------------- 17.8/351.2 MB 8.2 MB/s 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.51.0 requires protobuf<7,>=3.20, but you have protobuf 7.34.1 which is incompatible.


In [4]:
pip install --upgrade pip

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------------------------- ---------- 1.3/1.8 MB 10.1 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 6.9 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [5]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [7]:
df = pd.read_csv(r"C:\Users\Sreeram\Downloads\imbalanced_data.csv")  

df = df[['tweet', 'label']]
df.head()

,tweet,label
0,@user when a father is dysfunctional and is s...,0
1,@user @user thanks for #lyft credit i can't us...,0
2,bihday your majesty,0
3,#model i love u take with u all the time in ...,0
4,factsguide: society now #motivation,0


In [9]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'@\w+', '', text)        
    text = re.sub(r'http\S+', '', text)      
    text = re.sub(r'[^a-z\s]', '', text)      
    text = re.sub(r'\s+', ' ', text).strip()  
    return text

df['clean_tweet'] = df['tweet'].apply(clean_text)

In [10]:
vocab_size = 5000

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(df['clean_tweet'])

sequences = tokenizer.texts_to_sequences(df['clean_tweet'])

In [11]:
max_len = 25  # between 20–30 as mentioned

X = pad_sequences(sequences, maxlen=max_len, padding='post')
y = df['label']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [13]:
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64, input_length=max_len),
    SimpleRNN(64),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

C:\Users\Sreeram\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn (SimpleRNN)               │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [14]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/10
640/640 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.9321 - loss: 0.2445 - val_accuracy: 0.9366 - val_loss: 0.2262
Epoch 2/10
640/640 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.9413 - loss: 0.1987 - val_accuracy: 0.9394 - val_loss: 0.2135
Epoch 3/10
640/640 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.9542 - loss: 0.1436 - val_accuracy: 0.9419 - val_loss: 0.1960
Epoch 4/10
640/640 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.9695 - loss: 0.0976 - val_accuracy: 0.9343 - val_loss: 0.2074
Epoch 5/10
640/640 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.9802 - loss: 0.0632 - val_accuracy: 0.9318 - val_loss: 0.2332
Epoch 6/10
640/640 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.9865 - loss: 0.0446 - val_accuracy: 0.9384 - val_loss: 0.2194
Epoch 7/10
640/640 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9907 - loss: 0.0336 - val_accuracy: 0.9355 - val_loss: 0.2422
Epoch 8/10
640/640 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.9926 - loss: 0.0264 - val_accu

In [15]:
y_pred = (model.predict(X_test) > 0.5).astype("int32")

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Accuracy: 0.9319568277803848

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.96      0.96      5945
           1       0.51      0.53      0.52       448

    accuracy                           0.93      6393
   macro avg       0.74      0.75      0.74      6393
weighted avg       0.93      0.93      0.93      6393


Confusion Matrix:
[[5719  226]
 [ 209  239]]


# Hate Speech Classification using RNN

## Problem Statement
### Build a binary classifier to detect hate speech from tweets using Simple RNN.

## Data Preprocessing
- Lowercasing
- Removed mentions, URLs
- Removed special characters and numbers
- Removed extra spaces

## Vocabulary Creation
- Used Keras Tokenizer
- Vocabulary size limited to 5000
- OOV token added

## Model Architecture
- Embedding Layer
- Simple RNN Layer
- Dense Output Layer (Sigmoid)

## Training Details
- Loss: Binary Crossentropy
- Optimizer: Adam
- Epochs: 10
- Batch Size: 32

## Results
- Accuracy: XX%